# Representative Rainfall Station Generator (Modules A-F)

Generates representative rainfall stations from IMD 0.25° gridded daily rainfall
for any basin in India (Hybrid method: area -> distance -> correlation -> K-Means/Silhouette).

**How to use**
1. Run the *Setup* cell (installs geopandas etc — takes ~1 min, first time only)
2. Run the *Upload basin shapefile* cell — upload a **zip** containing `.shp .dbf .shx .prj`
3. Run the *Get IMD .grd data* cell — either upload a zip, or mount Google Drive and point to the zip path
4. Adjust filter settings in the *Parameters* cell if needed
5. Run remaining cells in order — final tables/plots display inline, and download buttons appear at the end


## 1. Setup

In [ ]:
!pip -q install geopandas shapely scikit-learn openpyxl

import os, re, glob, calendar, zipfile, tempfile, io
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import matplotlib.pyplot as plt

# IMD grid spec
LON_START, LAT_START = 66.5, 6.5
RES = 0.25
NCOLS, NROWS = 135, 129
CELLS_PER_DAY = NCOLS * NROWS
MISSING_VAL = -999.0

print("Setup complete.")


## 2. Parameters (edit if needed)

In [ ]:
MIN_GRID_AREA_KM2 = 30        # Module A: remove grids smaller than this
DISTANCE_THRESH_KM = 30       # Module C: min separation between stations
CORR_THRESH = 0.95            # Module D: drop station if r > this with a kept station
MIN_REPRESENTED_AREA_KM2 = 500  # Module F: drop final stations below this area

# Module E clustering
CLUSTER_MODE = "auto"   # "auto" (Silhouette search) or "manual"
K_RANGE = (10, 25)      # used if CLUSTER_MODE == "auto"
MANUAL_K = 12           # used if CLUSTER_MODE == "manual"

# Year range for IMD extraction (set after you see available years in Module B)
START_YEAR = None
END_YEAR = None


## 3. Upload basin shapefile (zip with .shp/.dbf/.shx/.prj)

In [ ]:
from google.colab import files

print("Select the basin shapefile ZIP...")
uploaded = files.upload()
basin_zip_path = list(uploaded.keys())[0]

tmpdir_basin = tempfile.mkdtemp()
with zipfile.ZipFile(basin_zip_path) as z:
    z.extractall(tmpdir_basin)

shp_files = glob.glob(os.path.join(tmpdir_basin, "**", "*.shp"), recursive=True)
assert shp_files, "No .shp file found in the uploaded zip."

basin_gdf = gpd.read_file(shp_files[0])
basin_gdf_wgs = basin_gdf.to_crs("EPSG:4326")
basin_union = basin_gdf_wgs.union_all()

print(f"Loaded basin: {len(basin_gdf_wgs)} feature(s), bounds = {basin_union.bounds}")


## 4. Get IMD .grd data

Pick ONE of the two options below.

- **Option A (upload zip)** — fine for a few years
- **Option B (Google Drive)** — recommended for large multi-year archives. Mount Drive,
  then set `imd_zip_path` to the path of the zip inside your Drive


In [ ]:
# ---- OPTION A: upload a zip directly ----
USE_DRIVE = False  # set True to use Option B instead

if not USE_DRIVE:
    print("Select the IMD .grd ZIP (one .grd file per year)...")
    uploaded_imd = files.upload()
    imd_zip_path = list(uploaded_imd.keys())[0]


In [ ]:
# ---- OPTION B: Google Drive ----
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    # EDIT this path to point to your zip inside Drive:
    imd_zip_path = "/content/drive/MyDrive/IMD_1984-2014_.zip"


In [ ]:
grd_dir = tempfile.mkdtemp()
with zipfile.ZipFile(imd_zip_path) as z:
    z.extractall(grd_dir)

grd_files = glob.glob(os.path.join(grd_dir, "**", "*.grd"), recursive=True)
grd_files_by_year = {}
for f in grd_files:
    m = re.search(r"(\d{4})", os.path.basename(f))
    if m:
        grd_files_by_year[int(m.group(1))] = f

available_years = sorted(grd_files_by_year.keys())
print(f"Found {len(available_years)} year(s): {available_years[0]}-{available_years[-1]}")

if START_YEAR is None:
    START_YEAR = available_years[0]
if END_YEAR is None:
    END_YEAR = available_years[-1]

print(f"Using year range: {START_YEAR}-{END_YEAR}")


## MODULE A — Generate IMD grid centroids & find grids inside basin

In [ ]:
def generate_grid_cells():
    records = []
    half = RES / 2
    for j in range(NCOLS):
        lon = LON_START + j * RES
        for i in range(NROWS):
            lat = LAT_START + i * RES
            grid_id = f"R{i:03d}C{j:03d}"
            geom = box(lon - half, lat - half, lon + half, lat + half)
            records.append({"Grid_ID": grid_id, "row": i, "col": j,
                             "Longitude": lon, "Latitude": lat, "geometry": geom})
    return gpd.GeoDataFrame(records, geometry="geometry", crs="EPSG:4326")


def equal_area_crs(basin_gdf_wgs):
    cen = basin_gdf_wgs.union_all().centroid
    zone = int((cen.x + 180) / 6) + 1
    return f"EPSG:{32600 + zone}"


grid_gdf = generate_grid_cells()
metric_crs = equal_area_crs(basin_gdf_wgs)

candidates = grid_gdf[grid_gdf.intersects(basin_union)].copy()
candidates_m = candidates.to_crs(metric_crs)
basin_m = gpd.GeoDataFrame(geometry=[basin_union], crs="EPSG:4326").to_crs(metric_crs)
basin_geom_m = basin_m.union_all()

candidates["Represented_Area_km2"] = [
    geom.intersection(basin_geom_m).area / 1e6 for geom in candidates_m.geometry
]
after_area = candidates[candidates["Represented_Area_km2"] >= MIN_GRID_AREA_KM2].copy()

print(f"Total IMD grids: {len(grid_gdf)}")
print(f"Grids intersecting basin: {len(candidates)}")
print(f"After area filter (>= {MIN_GRID_AREA_KM2} km2): {len(after_area)}")
print(f"Total represented area: {after_area['Represented_Area_km2'].sum():,.1f} km2")


## MODULE B — Read IMD .grd files for selected years

In [ ]:
def days_in_year(year):
    return 366 if calendar.isleap(year) else 365


def read_grd_year(filepath, year, grid_rows, grid_cols):
    expected_days = days_in_year(year)
    data = np.fromfile(filepath, dtype=np.float32)
    actual_days = data.size // CELLS_PER_DAY
    if actual_days != expected_days:
        raise ValueError(f"{os.path.basename(filepath)}: expected {expected_days} days, got {actual_days}.")
    arr = data.reshape(actual_days, NROWS, NCOLS)
    series = arr[:, grid_rows, grid_cols]
    return np.where(series == MISSING_VAL, np.nan, series)


grid_rows = after_area["row"].values
grid_cols = after_area["col"].values
grid_ids = after_area["Grid_ID"].values

all_dates, all_data = [], []
for year in range(START_YEAR, END_YEAR + 1):
    if year not in grd_files_by_year:
        raise FileNotFoundError(f"No .grd file found for year {year}.")
    series = read_grd_year(grd_files_by_year[year], year, grid_rows, grid_cols)
    n_days = series.shape[0]
    all_dates.extend(pd.date_range(start=f"{year}-01-01", periods=n_days, freq="D"))
    all_data.append(series)
    print(f"  {year}: {n_days} days read OK")

daily_df = pd.DataFrame(np.vstack(all_data), columns=grid_ids)
daily_df.insert(0, "Date", all_dates)

print(f"\nDaily rainfall matrix: {daily_df.shape[0]} days x {daily_df.shape[1]-1} grids")
daily_df.head()


## MODULE C — Distance filter

In [ ]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))


def distance_filter(stations_df, thresh_km):
    df = stations_df.sort_values("Represented_Area_km2", ascending=False).reset_index(drop=True)
    kept = []
    for _, row in df.iterrows():
        too_close = any(
            haversine_km(row["Latitude"], row["Longitude"], k["Latitude"], k["Longitude"]) < thresh_km
            for k in kept
        )
        if not too_close:
            kept.append(row)
    return pd.DataFrame(kept).reset_index(drop=True)


after_distance = distance_filter(after_area, DISTANCE_THRESH_KM)
print(f"After distance filter (>= {DISTANCE_THRESH_KM} km apart): {len(after_distance)} stations")


## MODULE D — Correlation filter

In [ ]:
def correlation_filter(stations_df, daily_df, thresh):
    df = stations_df.sort_values("Represented_Area_km2", ascending=False).reset_index(drop=True)
    kept_rows, kept_ids = [], []
    for _, row in df.iterrows():
        gid = row["Grid_ID"]
        too_corr = any(daily_df[gid].corr(daily_df[kid]) > thresh for kid in kept_ids)
        if not too_corr:
            kept_rows.append(row)
            kept_ids.append(gid)
    return pd.DataFrame(kept_rows).reset_index(drop=True)


after_corr = correlation_filter(after_distance, daily_df, CORR_THRESH)
print(f"After correlation filter (r > {CORR_THRESH}): {len(after_corr)} stations")


## MODULE E — Rainfall statistics + K-Means clustering

In [ ]:
def compute_rainfall_stats(daily_df, grid_ids):
    df = daily_df.copy()
    df["Year"] = df["Date"].dt.year
    stats = {}
    for gid in grid_ids:
        series = df[gid]
        annual = df.groupby("Year")[gid].sum(min_count=1)
        stats[gid] = {
            "Mean_Annual_Rainfall_mm": annual.mean(),
            "Std_Dev_mm": series.std(),
            "CV": series.std() / series.mean() if series.mean() != 0 else np.nan,
            "Max_Daily_Rainfall_mm": series.max(),
            "Wet_Days": int((series > 1.0).sum()),
        }
    return pd.DataFrame.from_dict(stats, orient="index").reset_index().rename(columns={"index": "Grid_ID"})


stats_df = compute_rainfall_stats(daily_df, after_corr["Grid_ID"].tolist())

feature_cols = ["Mean_Annual_Rainfall_mm", "Std_Dev_mm", "CV", "Max_Daily_Rainfall_mm", "Wet_Days"]
X_scaled = StandardScaler().fit_transform(stats_df[feature_cols].values)

if CLUSTER_MODE == "auto":
    min_k, max_k = K_RANGE
    max_k = min(max_k, len(stats_df) - 1)
    best_k, best_score, best_labels, scores = None, -1, None, {}
    for k in range(min_k, max_k + 1):
        labels = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X_scaled)
        score = silhouette_score(X_scaled, labels)
        scores[k] = score
        if score > best_score:
            best_score, best_k, best_labels = score, k, labels
    print(f"Optimal k (Silhouette) = {best_k}  (score={best_score:.3f})")
    for k, s in scores.items():
        print(f"  k={k}: {s:.4f}" + ("  <-- best" if k == best_k else ""))
else:
    best_k = MANUAL_K
    best_labels = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit_predict(X_scaled)
    print(f"Manual k = {best_k}")

stats_df["Cluster_ID"] = best_labels


## MODULE F — Select representative station per cluster + final export

In [ ]:
merged = stats_df.merge(after_corr, on="Grid_ID")
reps = []
for cid, grp in merged.groupby("Cluster_ID"):
    rep = grp.sort_values("Represented_Area_km2", ascending=False).iloc[0]
    reps.append(rep)
final = pd.DataFrame(reps).sort_values("Cluster_ID").reset_index(drop=True)

final_kept = final[final["Represented_Area_km2"] >= MIN_REPRESENTED_AREA_KM2].reset_index(drop=True)
final_dropped = final[final["Represented_Area_km2"] < MIN_REPRESENTED_AREA_KM2]

final_kept = final_kept.sort_values("Represented_Area_km2", ascending=False).reset_index(drop=True)
final_kept.insert(0, "Station_ID", [f"ST{i+1:03d}" for i in range(len(final_kept))])

output_cols = ["Station_ID", "Grid_ID", "Latitude", "Longitude", "Cluster_ID", "Represented_Area_km2",
                "Mean_Annual_Rainfall_mm", "Std_Dev_mm", "CV", "Max_Daily_Rainfall_mm", "Wet_Days"]
final_table = final_kept[output_cols]

print("=== PIPELINE SUMMARY ===")
print(f"Total IMD Grids: {len(grid_gdf)}")
print(f"Grids Inside Basin: {len(candidates)}")
print(f"After Area Filter: {len(after_area)}")
print(f"After Distance Filter: {len(after_distance)}")
print(f"After Correlation Filter: {len(after_corr)}")
print(f"After Clustering: {len(final)}")
print(f"Final Representative Stations: {len(final_table)}")
if len(final_dropped) > 0:
    print(f"\nDropped (area < {MIN_REPRESENTED_AREA_KM2} km2): {final_dropped['Grid_ID'].tolist()}")

final_table


## Plot: basin outline + final station locations

In [ ]:
fig, ax = plt.subplots(figsize=(7, 7))
basin_gdf_wgs.boundary.plot(ax=ax, color="black", linewidth=1)
ax.scatter(final_table["Longitude"], final_table["Latitude"], c="red", s=60, zorder=5)
for _, r in final_table.iterrows():
    ax.annotate(r["Station_ID"], (r["Longitude"], r["Latitude"]), xytext=(3, 3), textcoords="offset points", fontsize=8)
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_title("Final representative rainfall stations")
plt.tight_layout()
plt.show()


## Build daily / monthly rainfall outputs

In [ ]:
rename_map = dict(zip(final_kept["Grid_ID"], final_kept["Station_ID"]))
rep_daily = daily_df[["Date"] + list(rename_map.keys())].rename(columns=rename_map)

rep_monthly = rep_daily.copy()
rep_monthly["Year"] = rep_monthly["Date"].dt.year
rep_monthly["Month"] = rep_monthly["Date"].dt.month
monthly = rep_monthly.drop(columns="Date").groupby(["Year", "Month"]).sum(min_count=1).reset_index()

station_cols = list(rename_map.values())
basin_monthly = monthly[["Year", "Month"]].copy()
basin_monthly["Basin_Rainfall_mm"] = monthly[station_cols].mean(axis=1)

print(f"Daily: {rep_daily.shape}, Monthly: {monthly.shape}, Basin monthly: {basin_monthly.shape}")
rep_daily.head()


## Download results

In [ ]:
def save_excel(df, name):
    path = f"/content/{name}"
    df.to_excel(path, index=False)
    return path

paths = [
    save_excel(final_table, "Final_Stations.xlsx"),
    save_excel(rep_daily, "Representative_Stations_Rainfall.xlsx"),
    save_excel(monthly, "Representative_Stations_Monthly.xlsx"),
    save_excel(basin_monthly, "Basin_Monthly_Rainfall.xlsx"),
]

for p in paths:
    files.download(p)
